# Buổi 3 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5).

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Bấm giờ ba công cụ

In [ ]:
%run bam_gio.py

## Bước 2 — Đếm theo giờ bằng code có sẵn, hỏi năm câu

In [ ]:
%run thoi_gian.py

In [ ]:
from thoi_gian import dem_chuyen_theo_gio, doc_chuyen_taxi

for thang, ngay in [("03", "2024-03-10"), ("11", "2024-11-03")]:   # hai ngày đổi giờ ở New York
    dem = dem_chuyen_theo_gio(doc_chuyen_taxi(thang)).set_index("ds")
    print(dem.loc[f"{ngay} 00:00":f"{ngay} 05:00", "y"], "\n")

In [ ]:
import pandas as pd

import tv

t = pd.read_parquet(tv.THU_MUC_DU_LIEU / "nyc-tlc-yellow-2024-11" / "yellow_tripdata_2024-11.parquet",
                    columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
don, tra = t["tpep_pickup_datetime"], t["tpep_dropoff_datetime"]
print(don.dtype, don.dt.tz)                                           # 1. kiểu, múi giờ
print(don.min(), don.max())                                           # 2. khoảng thời gian
print((~don.between("2024-11-01", "2024-12-01", inclusive="left")).sum())   # 3. số chuyến ngoài tháng
print((tra < don).sum())                                              # 4. số chuyến thời lượng âm
gio = don.dt.floor("h").drop_duplicates()                             # mỗi giờ có dữ liệu giữ một lần
gio[gio.between("2024-11-01", "2024-12-01", inclusive="left")].dt.date.value_counts().value_counts()   # 5.

## Bước 4 — Sửa `chuan_hoa_thoi_gian`

Sửa trong `thoi_gian.py` (tài liệu mục 5, bước 4), chạy lại ô bước 2 để xem giờ 0 chuyến và giờ gấp đôi đã hết.
Chấm: `python lab.py check` trong terminal.

## Bước 5 — Dạng dài theo khu vực (mục 4.5)

In [ ]:
from thoi_gian import chuan_hoa_thoi_gian

chuyen = doc_chuyen_taxi("03")
dai = chuan_hoa_thoi_gian(chuyen.assign(mot=1), "tpep_pickup_datetime", "mot", "h",
                          mui_gio_nguon="America/New_York", cot_id="PULocationID", gop="sum",
                          mo_ho="NaT", khoang=("2024-03-01 05:00", "2024-04-01 04:00"))
print(dai.groupby("unique_id").size().unique())         # số dòng mỗi chuỗi: phải là [743]
print(dai.duplicated(["unique_id", "ds"]).sum())         # phải là 0
print((dai["y"] == 0).mean().round(3))                   # tỷ lệ ô bằng 0